# Phân tích câu truy vấn sản phẩm thời trang (áo/quần/giày)

Notebook đọc `dataset_truyvan_sanpham_500.csv`, trích xuất thuộc tính như danh mục (category), giới tính, tay áo, mùa, form, và giá (dưới/trên/khoảng/từ-đến/xấp xỉ).


In [1]:
# Đọc dataset truy vấn
import pandas as pd

CSV_PATH = r"C:\Users\Admin\Desktop\cl\test_notebook\dataset_truyvan_sanpham_500.csv"
df_queries = pd.read_csv(CSV_PATH)
df_queries.head(8)


,id,text
0,1,Gợi ý cho tôi vài mẫu áo croptop dành cho nữ g...
1,2,Gợi ý cho tôi vài mẫu quần jean dài tay giá da...
2,3,Tôi muốn xem đầm dành cho nam giá khoảng dưới ...
3,4,Hãy tìm cho tôi quần jean mặc mùa hè giá trong...
4,5,Tôi muốn mua áo vest dành cho nữ khoảng tầm 3 ...
5,6,Gợi ý cho tôi vài mẫu giày cao gót form ôm giá...
6,7,Tìm áo blouse mặc đi chơi giá tầm 3 triệu giúp...
7,8,Tôi muốn mua áo sơ mi form rộng khoảng tầm 1 t...


In [2]:
# Từ điển đồng nghĩa và thuộc tính
import re
from typing import Optional, Tuple, Dict

CATEGORY_SYNONYMS = {
    "áo thun": {"áo thun", "tshirt", "tee"},
    "áo sơ mi": {"áo sơ mi", "sơ mi", "shirt"},
    "áo khoác": {"áo khoác", "khoác", "jacket"},
    "áo len": {"áo len", "len", "sweater"},
    "áo cardigan": {"áo cardigan", "cardigan"},
    "áo hoodie": {"áo hoodie", "hoodie"},
    "áo vest": {"áo vest", "vest", "blazer"},
    "áo croptop": {"áo croptop", "croptop"},
    "áo ba lỗ": {"áo ba lỗ", "ba lỗ", "tank top"},
    "áo polo": {"áo polo", "polo"},
    "quần jean": {"quần jean", "jean", "jeans", "denim"},
    "quần short": {"quần short", "short"},
    "quần tây": {"quần tây", "tây", "trousers", "slacks"},
    "váy": {"váy", "đầm", "dress"},
    "chân váy": {"chân váy", "váy ngắn", "skirt"},
    "giày sneaker": {"giày sneaker", "sneaker"},
    "giày cao gót": {"giày cao gót", "cao gót", "heels"},
    "giày da": {"giày da", "da", "leather shoes"},
}

GENDER_SYNONYMS = {
    "nam": {"nam", "dành cho nam", "men", "male"},
    "nữ": {"nữ", "dành cho nữ", "women", "female"},
    "unisex": {"unisex", "cả nam nữ"},
}

SLEEVE_SYNONYMS = {
    "ngắn tay": {"ngắn tay"},
    "dài tay": {"dài tay"},
}

SEASON_SYNONYMS = {
    "mùa hè": {"mùa hè", "hè", "mặc mùa hè"},
    "mùa đông": {"mùa đông", "đông", "mặc mùa đông"},
}

FIT_SYNONYMS = {
    "form rộng": {"form rộng", "rộng", "oversize", "oversized"},
    "form ôm": {"form ôm", "ôm", "slim", "fitted"},
}

COLOR_SYNONYMS = {
    "đen": {"đen", "black"},
    "trắng": {"trắng", "white"},
    "xám": {"xám", "grey", "gray"},
    "xanh": {"xanh", "blue", "xanh đậm", "navy"},
    "đỏ": {"đỏ", "red"},
}

_DEF_JOIN = lambda s: re.sub(r"\s+", " ", s.strip().lower())




In [3]:
# Phân tích giá nâng cao: dưới/trên/khoảng/từ-đến/xấp xỉ/dao động quanh
from typing import NamedTuple, Optional

class PriceQuery(NamedTuple):
    lower: Optional[int]
    upper: Optional[int]
    approx: Optional[int]
    mode: str  # one of: range, under, over, approx, none

VND = 1
K = 1_000
TRIEU = 1_000_000

_num = r"(\d+[\.,]?\d*)"

_patterns = [
    ("range", rf"trong tầm|trong khoảng|trong\s+khoảng|từ\s+{_num}\s+đến\s+{_num}"),
]

# Hàm chuẩn hóa số có đơn vị VND
_def_norm = _DEF_JOIN

def _parse_number_unit(text: str) -> Optional[int]:
    t = _def_norm(text)
    m = re.search(rf"{_num}\s*(triệu|tr|nghìn|k)?", t)
    if not m:
        return None
    val = float(m.group(1).replace(',', '.'))
    unit = m.group(2)
    if unit in {"triệu", "tr"}:
        return int(round(val * TRIEU))
    if unit in {"nghìn", "k"}:
        return int(round(val * K))
    # nếu không có đơn vị và số lớn, coi như VND
    if val >= 1000:
        return int(val)
    # số nhỏ mà không có đơn vị -> giả định nghìn
    return int(round(val * K))


def parse_price_advanced(text: str) -> PriceQuery:
    t = _def_norm(text)
    # từ-đến
    m = re.search(rf"từ\s+{_num}(?:\s*(triệu|tr|nghìn|k))?\s+đến\s+{_num}(?:\s*(triệu|tr|nghìn|k))?", t)
    if m:
        n1 = _parse_number_unit(m.group(1) + (" " + (m.group(2) or "") if m.group(2) else ""))
        n2 = _parse_number_unit(m.group(3) + (" " + (m.group(4) or "") if m.group(4) else ""))
        if n1 and n2:
            lo, hi = sorted([n1, n2])
            return PriceQuery(lower=lo, upper=hi, approx=None, mode="range")

    # dưới / dưới hoặc bằng
    m = re.search(rf"(dưới|<=?|ít hơn)\s+{_num}(?:\s*(triệu|tr|nghìn|k))?", t)
    if m:
        n = _parse_number_unit(m.group(2) + (" " + (m.group(3) or "") if m.group(3) else ""))
        if n:
            return PriceQuery(lower=None, upper=n, approx=None, mode="under")

    # trên / lớn hơn
    m = re.search(rf"(trên|>=?|lớn hơn)\s+{_num}(?:\s*(triệu|tr|nghìn|k))?", t)
    if m:
        n = _parse_number_unit(m.group(2) + (" " + (m.group(3) or "") if m.group(3) else ""))
        if n:
            return PriceQuery(lower=n, upper=None, approx=None, mode="over")

    # khoảng/tầm/xấp xỉ/dao động quanh ~ ±20%
    m = re.search(rf"(khoảng|tầm|xấp xỉ|dao động quanh)\s+{_num}(?:\s*(triệu|tr|nghìn|k))?", t)
    if m:
        n = _parse_number_unit(m.group(2) + (" " + (m.group(3) or "") if m.group(3) else ""))
        if n:
            return PriceQuery(lower=int(n * 0.8), upper=int(n * 1.2), approx=n, mode="approx")

    # số trần
    n = _parse_number_unit(t)
    if n:
        return PriceQuery(lower=int(n * 0.9), upper=int(n * 1.1), approx=n, mode="approx")

    return PriceQuery(lower=None, upper=None, approx=None, mode="none")


In [4]:
# Hàm tiện ích tìm khóa chuẩn theo synonyms
from typing import Iterable

def _find_key_by_synonyms(text: str, synonyms_map: Dict[str, Iterable[str]]) -> Optional[str]:
    t = _DEF_JOIN(text)
    for canonical, variants in synonyms_map.items():
        for v in variants:
            if re.search(rf"\b{re.escape(v)}\b", t):
                return canonical
    return None




In [5]:
# Extract đầy đủ thuộc tính từ câu truy vấn
from dataclasses import dataclass
from typing import List

@dataclass
class ApparelQuery:
    category: Optional[str]
    gender: Optional[str]
    sleeve: Optional[str]
    season: Optional[str]
    fit: Optional[str]
    color: Optional[str]
    price: PriceQuery


def extract_apparel_attributes(text: str) -> ApparelQuery:
    t = _DEF_JOIN(text)
    category = _find_key_by_synonyms(t, CATEGORY_SYNONYMS)
    gender = _find_key_by_synonyms(t, GENDER_SYNONYMS)
    sleeve = _find_key_by_synonyms(t, SLEEVE_SYNONYMS)
    season = _find_key_by_synonyms(t, SEASON_SYNONYMS)
    fit = _find_key_by_synonyms(t, FIT_SYNONYMS)
    color = _find_key_by_synonyms(t, COLOR_SYNONYMS)
    price = parse_price_advanced(t)
    return ApparelQuery(category, gender, sleeve, season, fit, color, price)

# Thử nhanh trên một câu
extract_apparel_attributes("Tìm áo khoác dài tay dành cho nam giá trong khoảng 1 đến 2 triệu")


ApparelQuery(category='áo khoác', gender='nam', sleeve='dài tay', season=None, fit=None, color=None, price=PriceQuery(lower=800, upper=1200, approx=1000, mode='approx'))

In [6]:
# Demo áp dụng lên 10 dòng đầu và hiển thị DataFrame

def _price_to_str(p: PriceQuery) -> str:
    if p.mode == "range":
        return f"[{p.lower:,}..{p.upper:,}]".replace(",", ".")
    if p.mode == "under":
        return f"<= {p.upper:,}".replace(",", ".")
    if p.mode == "over":
        return f">= {p.lower:,}".replace(",", ".")
    if p.mode == "approx":
        return f"~{p.approx:,} (~{p.lower:,}-{p.upper:,})".replace(",", ".")
    return ""

rows = []
for _, r in df_queries.head(10).iterrows():
    pq = extract_apparel_attributes(r["text"])
    rows.append({
        "id": r["id"],
        "text": r["text"],
        "category": pq.category,
        "gender": pq.gender,
        "sleeve": pq.sleeve,
        "season": pq.season,
        "fit": pq.fit,
        "color": pq.color,
        "price": _price_to_str(pq.price),
    })

pd.DataFrame(rows)


,id,text,category,gender,sleeve,season,fit,color,price
0,1,Gợi ý cho tôi vài mẫu áo croptop dành cho nữ g...,áo croptop,nữ,None,None,None,None,~1.000.000 (~800.000-1.200.000)
1,2,Gợi ý cho tôi vài mẫu quần jean dài tay giá da...,quần jean,None,dài tay,None,None,None,~1.000.000 (~800.000-1.200.000)
2,3,Tôi muốn xem đầm dành cho nam giá khoảng dưới ...,váy,nam,None,None,None,None,<= 500.000
3,4,Hãy tìm cho tôi quần jean mặc mùa hè giá trong...,quần jean,None,None,mùa hè,None,None,~1.000 (~800-1.200)
4,5,Tôi muốn mua áo vest dành cho nữ khoảng tầm 3 ...,áo vest,nữ,None,None,None,None,~3.000.000 (~2.400.000-3.600.000)
5,6,Gợi ý cho tôi vài mẫu giày cao gót form ôm giá...,giày cao gót,None,None,None,form ôm,None,<= 1.000.000
6,7,Tìm áo blouse mặc đi chơi giá tầm 3 triệu giúp...,None,None,None,None,None,None,~3.000.000 (~2.400.000-3.600.000)
7,8,Tôi muốn mua áo sơ mi form rộng khoảng tầm 1 t...,áo sơ mi,None,None,None,form rộng,None,~1.000.000 (~800.000-1.200.000)
8,9,Gợi ý cho tôi vài mẫu áo blouse mặc mùa đông g...,None,None,None,mùa đông,None,None,<= 1.000.000
9,10,Tôi muốn mua quần short mặc mùa đông khoảng da...,quần short,None,None,mùa đông,None,None,~1.000.000 (~800.000-1.200.000)
